In [ ]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


# Kvasir-VQA x1 — BLIP VQA baseline

Evaluate BLIP VQA (zero-shot) on Kvasir-VQA x1 metadata. Computes BLEU/ROUGE-L on a sample and yes/no accuracy on the yes/no subset.

In [1]:
import os
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import BlipProcessor, BlipForQuestionAnswering
from datasets import load_dataset
import evaluate



import os, pathlib, subprocess

p = pathlib.Path("~/.cache/huggingface").expanduser()
print("is_symlink:", p.is_symlink())
print("link_points_to:", os.readlink(p) if p.is_symlink() else None)

real = pathlib.Path(os.path.realpath(p))
print("realpath:", real)
print("realpath_exists:", real.exists())
print("hub_exists:", (real / "hub").exists())

print("\nDF (will fail if not mounted):")
try:
    print(subprocess.check_output(["df","-h", str(real)]).decode())
except Exception as e:
    print("df failed:", e)

print("\n/media/arcturus contents:")
print(subprocess.check_output(["bash","-lc","ls -la /media/arcturus || true"]).decode())



2026-01-22 13:12:32.273827: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


is_symlink: True
link_points_to: /mnt/hf/huggingface-cache
realpath: /mnt/hf/huggingface-cache
realpath_exists: True
hub_exists: True

DF (will fail if not mounted):
Filesystem      Size  Used Avail Use% Mounted on
/dev/sdb1       1.8T  1.2T  583G  67% /mnt/hf


/media/arcturus contents:
total 24
drwxr-x---+ 4 root     root      4096 Jan 22 10:17 .
drwxr-xr-x  4 root     root      4096 Aug 29  2024 ..
drwxr-xr-x  3 root     root      4096 Jan  3 03:20 34d31dbb-6968-420e-aeb8-7d2f0629b913
drwxrwxrwx  1 arcturus arcturus 12288 Dec 29 12:17 7060ED2760ECF52E



In [ ]:
# Paths & config
HF_DATASET = "SimulaMet-HOST/Kvasir-VQA"  # adjust if using a different HF source
def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. \n"
        "Run this notebook from within the Kvasir_VQA_x1 folder, \n"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "blip_baseline" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Salesforce/blip-vqa-base"
SAMPLE_N = None  # set None for all rows (can be slow)
SAMPLE_YN = None  # yes/no subset sample
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/blip_baseline/out
Device: cuda


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)
print("Rows:", len(meta))
print(meta.head())

# Use full dataset
# If you want to restrict to a split, reintroduce split filtering here.
dev_df = meta.copy()
print("Dev rows:", len(dev_df))


Rows: 58849
   split                     img_id  \
0  train  cla820gl0s3nv071u4fgd7xgq   
1  train  cla820gl0s3nv071u4fgd7xgq   
2  train  cla820gl0s3nv071u4fgd7xgq   
3  train  cla820gl0s3nv071u4fgd7xgq   
4  train  cla820gl0s3nv071u4fgd7xgq   

                                     image_path  \
0  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
1  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
2  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
3  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   
4  out/images/raw/cla820gl0s3nv071u4fgd7xgq.jpg   

                                            question              answer  \
0  Are there any abnormalities in the image? Chec...  ulcerative colitis   
1  Are there any anatomical landmarks in the imag...                none   
2  Are there any instruments in the image? Check ...                none   
3                      Have all polyps been removed?        not relevant   
4                    Is this finding easy to detect?              

In [4]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()
print("Loaded BLIP")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded BLIP


In [5]:
# Fallback: load HF dataset and map img_id -> image if local file missing
HF_IMG_MAP = None
HF_DATASET_CACHE = None

def load_fallback_image(img_id: str):
    global HF_IMG_MAP, HF_DATASET_CACHE
    if HF_IMG_MAP is None:
        # try to load from HF
        try:
            ds_all = load_dataset(HF_DATASET)
            # pick first split
            if 'raw' in ds_all:
                ds_use = ds_all['raw']
            elif 'train' in ds_all:
                ds_use = ds_all['train']
            else:
                ds_use = list(ds_all.values())[0]
            HF_DATASET_CACHE = ds_use
            HF_IMG_MAP = {}
            for ex in ds_use:
                iid = ex.get('img_id') or ex.get('image_id') or ex.get('id') or ex.get('filename')
                if iid is None:
                    continue
                iid = str(iid).split('/')[-1].split('.')[0]
                HF_IMG_MAP[iid] = ex.get('image')
            print("Built HF image map:", len(HF_IMG_MAP))
        except Exception as e:
            print("Fallback HF load failed:", e)
            HF_IMG_MAP = {}
    return HF_IMG_MAP.get(str(img_id))


In [6]:
def generate_answer(row):
    img_path = Path(row["image_path"])
    img = None
    if img_path.is_file():
        img = Image.open(img_path).convert("RGB")
    else:
        img_id = row.get("img_id") or img_path.stem
        img = load_fallback_image(img_id)
        if img is None:
            raise FileNotFoundError(f"Image not found locally and HF fallback missing for {img_path}")
        if img.mode != "RGB":
            img = img.convert("RGB")
    q = str(row["question"])
    inputs = processor(images=img, text=q, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20)
    ans = processor.decode(out[0], skip_special_tokens=True)
    return ans

# BLEU/ROUGE metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

def eval_sample(df, n=None):
    if n is not None:
        df = df.sample(min(n, len(df)), random_state=SEED).reset_index(drop=True)
    preds = []
    refs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="BLIP eval"):
        ans = generate_answer(row)
        preds.append(ans)
        refs.append(str(row["answer"]))
    bleu_refs = [[r] for r in refs]  # sacrebleu expects list-of-list references
    bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
    rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"]
    return preds, refs, bleu_score, rouge_l


In [7]:
# Run eval on a sample for BLEU/ROUGE
preds, refs, bleu_score, rouge_l = eval_sample(dev_df, n=SAMPLE_N)

dev_df_sampled = dev_df.head(len(preds)).copy()
dev_df_sampled["pred_blip"] = preds

print("BLEU:", bleu_score)
print("ROUGE-L:", rouge_l)

# Save predictions
pred_path = OUT_DIR / "predictions_blip_sample.csv"
dev_df_sampled.to_csv(pred_path, index=False)
print("Saved:", pred_path)


BLIP eval:   0%|          | 0/500 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

Built HF image map: 6500
BLEU: 0.0
ROUGE-L: 0.2622583249224117
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/blip_baseline/out/predictions_blip_sample.csv


In [8]:
# Yes/No subset accuracy
yn_df = dev_df[dev_df["answer"].astype(str).str.lower().isin(["yes","no"])]
if SAMPLE_YN is not None:
    yn_df = yn_df.sample(min(SAMPLE_YN, len(yn_df)), random_state=SEED).reset_index(drop=True)

def to_yesno(text):
    t = str(text).lower()
    if "yes" in t:
        return "yes"
    if "no" in t:
        return "no"
    return "no"

yn_preds = []
for _, row in tqdm(yn_df.iterrows(), total=len(yn_df), desc="BLIP yes/no"):
    ans = generate_answer(row)
    yn_preds.append(to_yesno(ans))

y_true = yn_df["answer"].str.lower().tolist()
acc_yn = (pd.Series(yn_preds) == pd.Series(y_true)).mean() if len(yn_df) else 0
print("Yes/No accuracy:", acc_yn, "(n=", len(yn_df), ")")

yn_df_out = yn_df.copy()
yn_df_out["pred_blip_yesno"] = yn_preds
yn_path = OUT_DIR / "predictions_blip_yesno.csv"
yn_df_out.to_csv(yn_path, index=False)
print("Saved yes/no preds:", yn_path)


BLIP yes/no:   0%|          | 0/500 [00:00<?, ?it/s]

Yes/No accuracy: 0.546 (n= 500 )
Saved yes/no preds: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/blip_baseline/out/predictions_blip_yesno.csv


## Notes
- Adjust `SAMPLE_N` / `SAMPLE_YN` to control runtime (set to None for full eval).
- BLEU/ROUGE are on the sampled set; yes/no accuracy is on the yes/no subset.
- Use results to decide if fine-tuning or constrained decoding is needed.
